In [ ]:
!pip install -q torch transformers sentence-transformers spacy pytextrank
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 38.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import torch
import spacy
import pytextrank
from transformers import T5Tokenizer, T5ForConditionalGeneration
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
nlp = spacy.load("en_core_web_sm")
nlp.add_pipe("textrank")
path = "Path_of_files_saved_in_your_laptop"
tokenizer = T5Tokenizer.from_pretrained(path)
model = T5ForConditionalGeneration.from_pretrained(path)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()
embedder = SentenceTransformer("all-MiniLM-L6-v2")

/usr/local/lib/python3.12/dist-packages


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
import numpy as np
def summary(text,maxi=8,lambda_param=0.7):
    doc = nlp(text)
    sentences = [sent.text.strip() for sent in doc.sents]
    if len(sentences) <= maxi:
        return " ".join(sentences)
    embeddings = embedder.encode(sentences)
    embeddings1 = embedder.encode([text])
    doc_sim = cosine_similarity(
        embeddings,
        embeddings1
    ).flatten()
    sentence = []
    idx = []
    idx1 = np.argmax(doc_sim)
    sentence.append(sentences[idx1])
    idx.append(idx1)
    while len(sentence) < maxi:
        scores = []
        for i in range(len(sentences)):
            if i in idx:
                scores.append(-1)
                continue
            new_sim = cosine_similarity(
                embeddings[i].reshape(1, -1),
                embeddings[idx]
            ).max()
            mmr = (
                lambda_param*doc_sim[i]-(1-lambda_param)*new_sim
            )
            scores.append(mmr)
        next_idx = np.argmax(scores)
        sentence.append(sentences[next_idx])
        idx.append(next_idx)
    return " ".join(sentence)

In [ ]:
def summarize(text):
    extracted = summary(text, maxi=8, lambda_param=0.7)
    inputs = tokenizer(
        "summarize: " + extracted,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)
    with torch.no_grad():
        output = model.generate(
            inputs["input_ids"],
            max_length=160,
            num_beams=4,
            no_repeat_ngram_size=3
        )
    return tokenizer.decode(output[0], skip_special_tokens=True)

In [ ]:
text = """
ARTIFICIAL INTELLIGENCE AND MACHINE LEARNING

Artificial Intelligence (AI) is a branch of computer science that focuses on creating machines capable of performing tasks that normally require human intelligence. These tasks include learning, reasoning, problem-solving, perception, language understanding, and decision-making. AI aims to simulate human intelligence using algorithms, data, and computational power.

Machine Learning (ML) is a subset of Artificial Intelligence. Instead of explicitly programming rules, machine learning systems learn patterns from data. The core idea behind ML is that systems can improve their performance over time as they are exposed to more data.

TYPES OF ARTIFICIAL INTELLIGENCE

Artificial Intelligence can be broadly classified into three types:

1. Narrow AI (Weak AI):
Narrow AI is designed to perform a specific task. Examples include voice assistants, recommendation systems, spam filters, and facial recognition systems. Most AI systems today fall under this category.

2. General AI (Strong AI):
General AI refers to machines that possess human-level intelligence and can perform any intellectual task that a human can. This type of AI does not yet exist.

3. Super AI:
Super AI would surpass human intelligence in all aspects, including creativity, problem-solving, and emotional understanding. This is currently theoretical.

MACHINE LEARNING CATEGORIES

Machine learning is divided into three major categories:

1. Supervised Learning:
In supervised learning, models are trained using labeled data. Each input has a corresponding correct output. Common algorithms include Linear Regression, Logistic Regression, Support Vector Machines, and Neural Networks. Applications include spam detection and price prediction.

2. Unsupervised Learning:
Unsupervised learning uses unlabeled data. The goal is to discover hidden patterns or structures. Common techniques include clustering and dimensionality reduction. Algorithms include K-Means and Principal Component Analysis (PCA).

3. Reinforcement Learning:
In reinforcement learning, an agent learns by interacting with an environment and receiving rewards or penalties. The objective is to maximize cumulative reward. This approach is widely used in robotics and game-playing systems.

DEEP LEARNING

Deep learning is a specialized subset of machine learning that uses artificial neural networks with multiple layers. These networks are inspired by the human brain. Deep learning has been responsible for breakthroughs in image recognition, speech recognition, and natural language processing.

Neural networks consist of:
- Input layers
- Hidden layers
- Output layers

Each neuron applies a weighted sum followed by an activation function such as ReLU, Sigmoid, or Softmax.

NATURAL LANGUAGE PROCESSING

Natural Language Processing (NLP) is a field of AI that enables machines to understand, interpret, and generate human language. Tasks include text classification, summarization, translation, and question answering.

Common NLP techniques:
- Tokenization
- Stemming and Lemmatization
- TF-IDF
- Word Embeddings
- Transformers

Transformers use attention mechanisms to capture contextual relationships between words. Models such as BERT, GPT, and T5 are transformer-based.

APPLICATIONS OF AI AND ML

AI and ML are used in many industries:
- Healthcare: disease prediction, medical imaging
- Finance: fraud detection, algorithmic trading
- Education: personalized learning systems
- Transportation: self-driving vehicles
- Manufacturing: predictive maintenance

ETHICS AND FUTURE OF AI

With the growth of AI, ethical concerns have become important. These include data privacy, bias in algorithms, job displacement, and transparency. Responsible AI development focuses on fairness, accountability, and explainability.

The future of AI includes advancements in general intelligence, human-AI collaboration, and increased automation. AI and ML will continue to shape technology and society in the coming decades.

"""
ans = summarize(text)
print("SUMMARY:\n", ans)

SUMMARY:
 ARTIFICIAL INTELLIGENCE AND MACHINE LEARNING Artificial Intelligence (AI) is a branch of computer science that focuses on creating machines capable of performing tasks that normally require human intelligence.
